# Lecture 03b: Visualizing Data: the code

Every code cell from the slides, in order. Run them top to bottom.


## 🧾 Mid-Quarter Feedback Summary

In [ ]:
# (on the slides, this code is hidden: only its output shows)
# Shared look for the survey charts (hidden on slides). Colors: blue / orange categorical,
# red-gray-blue diverging for agree/disagree.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 15, "axes.titlesize": 17, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": "#888", "axes.labelcolor": "#333",
                     "xtick.color": "#444", "ytick.color": "#444", "figure.facecolor": "white",
                     "axes.facecolor": "white", "savefig.facecolor": "white"})
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#b9b8b2"
survey = pd.read_csv("data/midquarter_survey_2026.csv")
N = len(survey)

import base64
from IPython.display import HTML
def show_gif(path, width="90%"):
    """Show an animated GIF inline (works in Jupyter AND in Quarto slides; no file path to break)."""
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    return HTML(f'<div style="text-align:center"><img src="data:image/gif;base64,{b64}" width="{width}"></div>')

## ⏱️ Workload: 3–5 hours in a typical week

In [ ]:
# (on the slides, this code is hidden: only its output shows)
order = ["Under 3", "3-5", "6-8", "9-11", "12 or more"]
typ = survey["hours_typical"].value_counts().reindex(order, fill_value=0)
hev = survey["hours_heaviest"].value_counts().reindex(order, fill_value=0)
wl  = survey["workload"].value_counts().reindex(["About right", "Too heavy", "Too light"])

fig, (a, b) = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1.7, 1]})
x, w = np.arange(len(order)), 0.38
a.bar(x - w/2 - 0.02, typ, w, color=BLUE, label="Typical week")
a.bar(x + w/2 + 0.02, hev, w, color=ORANGE, label="Heaviest week")
a.set_xticks(x, order); a.set_xlabel("Hours per week outside lecture & lab")
a.set_ylabel("Number of students"); a.legend(frameon=False); a.grid(axis="y", alpha=0.3)
b.pie(wl, colors=[BLUE, ORANGE, GRAY], startangle=90, counterclock=False,
      autopct=lambda v: f"{v:.0f}%" if v > 5 else "", pctdistance=0.7,
      wedgeprops={"edgecolor": "white", "linewidth": 2}, textprops={"fontsize": 15, "color": "white"})
b.legend([f"{k} ({v / N:.0%})" for k, v in wl.items()], loc="center left", bbox_to_anchor=(0.95, 0.5), frameon=False)
b.set_title("The workload is ...")
plt.tight_layout(); plt.show()

## 👍 How is it going? Fairness high, lectures lowest

In [ ]:
# (on the slides, this code is hidden: only its output shows)
items = {"grading_fair": "Grading & assessments feel fair",
         "quiz1_matched_class": "Quiz 1 reflected what we covered",
         "labs_help": "Labs help me understand",
         "learned_a_lot": "I have learned a lot so far",
         "lectures_help": "Lectures help me understand"}
lik = pd.DataFrame({lab: survey[col].value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0) * 100
                    for col, lab in items.items()}).T
colors = {1: "#c23b3a", 2: "#f0a3a2", 3: "#d6d5cf", 4: "#86b6ef", 5: "#2a78d6"}
names = {1: "Strongly disagree", 2: "Disagree", 3: "Neutral", 4: "Agree", 5: "Strongly agree"}
fig, ax = plt.subplots(figsize=(15, 5))
y = np.arange(len(lik))[::-1]
left = -(lik[1] + lik[2] + lik[3] / 2)          # center the neutral block on 0
for k in [1, 2, 3, 4, 5]:
    ax.barh(y, lik[k], left=left, color=colors[k], edgecolor="white", linewidth=2, label=names[k], height=0.62)
    left = left + lik[k]
agree = lik[4] + lik[5]
for yi, a in zip(y, agree):
    ax.text(left.max() + 2, yi, f"{a:.0f}% agree", va="center", fontsize=15, color="#333")
ax.set_yticks(y, lik.index); ax.axvline(0, color="#888", lw=1)
ax.set_xlim(-45, 112); ax.set_xticks([-40, -20, 0, 20, 40, 60, 80, 100], ["40%", "20%", "0", "20%", "40%", "60%", "80%", "100%"])
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.45, 1.16), frameon=False, fontsize=13)
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
plt.tight_layout(); plt.show()

## 🙋 What you'd like more of

In [ ]:
# (on the slides, this code is hidden: only its output shows)
picks = survey["want_more_of"].str.split(";").explode()
pct = (picks.value_counts() / N * 100).sort_values()
fig, ax = plt.subplots(figsize=(14, 4.8))
ax.barh(pct.index, pct.values, color=BLUE, height=0.62)
for yi, v in enumerate(pct.values):
    ax.text(v + 0.8, yi, f"{v:.0f}%", va="center", fontsize=15, color="#333")
ax.set_xlabel("% of students who picked it"); ax.set_xlim(0, 55)
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
plt.tight_layout(); plt.show()

## 🤖 AI: most of you study the result

In [ ]:
# (on the slides, this code is hidden: only its output shows)
use_order = ["Never", "Sometimes, then I study the result", "Usually, with little review", "Almost always"]
cred_order = ["Fine", "A bit unfair", "Very unfair"]
use = survey["ai_use"].value_counts(normalize=True).reindex(use_order, fill_value=0) * 100
cred = survey["ai_same_credit"].value_counts(normalize=True).reindex(cred_order, fill_value=0) * 100
fig, (a, b) = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1.25, 1]})
a.barh(["Never", "Sometimes, then\nI study the result", "Usually, with\nlittle review", "Almost always"][::-1],
       use.values[::-1], color=BLUE, height=0.6)
a.set_title("How do you use AI on HW and labs?")
b.barh(cred_order[::-1], cred.values[::-1], color=ORANGE, height=0.6)
b.set_title("Heavy AI users get the same credit.\nThis feels ...")
for ax, vals in ((a, use.values[::-1]), (b, cred.values[::-1])):
    for yi, v in enumerate(vals):
        ax.text(v + 1, yi, f"{v:.0f}%", va="center", fontsize=15, color="#333")
    ax.set_xlim(0, 80); ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
plt.tight_layout(); plt.show()

## 📈 Quiz 1: mean below median

In [ ]:
# (on the slides, this code is hidden: only its output shows)
# Quiz 1 scores (%), transcribed from the gradebook column
counts = {100: 12, 97: 21, 96: 1, 94: 23, 91: 13, 88: 11, 85: 8, 82: 7, 79: 5, 78: 1, 76: 5, 73: 4, 70: 2}
tail = [67, 64, 63, 61, 61, 43, 34, 34, 31, 28, 28, 25, 25]
quiz1 = np.array([v for v, c in counts.items() for _ in range(c)] + tail, dtype=float)
m, med = quiz1.mean(), np.median(quiz1)
fig, (ax, bx) = plt.subplots(2, 1, figsize=(15, 4.6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.hist(quiz1, bins=np.arange(22.5, 105, 5), color=BLUE, edgecolor="white", linewidth=1.5)
ax.axvline(m, color=ORANGE, lw=3, label=f"mean = {m:.1f}")
ax.axvline(med, color="#222", lw=3, ls="--", label=f"median = {med:.1f}")
ax.set_ylabel("Students"); ax.legend(loc="upper left", frameon=False)
ax.set_title(f"Quiz 1 scores (n = {len(quiz1)})")
bx.boxplot(quiz1, orientation="horizontal", widths=0.6, patch_artist=True,
           boxprops={"facecolor": "#cde2fb", "edgecolor": "#333"}, medianprops={"color": "#222", "lw": 2})
bx.set_yticks([]); bx.spines["left"].set_visible(False); bx.set_xlabel("Score (%)")
plt.tight_layout(); plt.show()

## Exercise: count frequencies by hand

In [ ]:
def count_frequencies(items):
    freq = {}
    for x in items:
        if x in freq:
            freq[x] += 1
        else:
            freq[x] = 1
        print(x, '->', freq)        # watch the dictionary grow
    return freq

data = ['Red', 'Blue', 'Red', 'Green', 'Blue', 'Red']
count_frequencies(data)

## `value_counts()` in action

In [ ]:
import pandas as pd

colors = pd.Series(['Red', 'Blue', 'Red', 'Green', 'Blue', 'Red', 'Blue', 'Blue'])
colors.value_counts()

In [ ]:
colors.value_counts(sort=False)

## Common mistake: `df.value_counts()`

In [ ]:
import pandas as pd
df = pd.DataFrame({"color": ["Red", "Blue", "Red", "Red"],
                   "size":  ["S", "S", "L", "S"]})
print(df["color"].value_counts(), "\n")
print(df.value_counts())

## In Python: add `normalize=True`

In [ ]:
colors.value_counts(normalize=True)

In [ ]:
colors.value_counts(normalize=True) * 100

## Percents in engineering: same percent, different n

In [ ]:
import pandas as pd
fabs = pd.DataFrame({"tested": [1000, 20], "passed": [950, 19]},
                    index=["Fab A", "Fab B"])
fabs["yield_%"] = 100 * fabs["passed"] / fabs["tested"]
fabs

## In Python

In [ ]:
colors.nunique()

In [ ]:
colors.unique()

In [ ]:
import numpy as np

np.unique(colors) #what is different here?

In [ ]:
np.unique(colors, return_counts=True) #counts with numpy!

## Counter-example: missing values

In [ ]:
s = pd.Series(["Red", "Blue", None, "Red"])
print(s.unique())
print(s.nunique())                # NaN skipped
print(s.nunique(dropna=False))    # NaN counted

## Histograms in Python: `plt.hist()`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
np.random.seed(0)
data = np.random.normal(100, 15, 200)
plt.hist(data, bins=20, edgecolor='black')
plt.title("Distribution of Test Scores")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

## Counter-example: the bins change the story

In [ ]:
# (on the slides, this code is hidden: only its output shows)
fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, nb_ in zip(axs, [5, 20, 100]):
    ax.hist(data, bins=nb_, edgecolor='black')
    ax.set_title(f"bins = {nb_}")
    ax.set_xlabel("Score")
axs[0].set_ylabel("Frequency")
plt.tight_layout(); plt.show()

## Anatomy of a box plot

In [ ]:
# (on the slides, this code is hidden: only its output shows)
# Anatomy of a box plot, drawn by hand so every piece is labeled (exclusive quartiles, as on the quiz)
import numpy as np, matplotlib.pyplot as plt
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 30])
q1, med, q3 = 3, 5.5, 8                      # lower half 1..5, upper half 6..30
iqr = q3 - q1; lo_f, hi_f = q1 - 1.5 * iqr, q3 + 1.5 * iqr
lo_w, hi_w = x[x >= lo_f].min(), x[x <= hi_f].max()
fig, ax = plt.subplots(figsize=(15, 5.2))
B, D, G = "#2a78d6", "#1f3b63", "#888"
ax.add_patch(plt.Rectangle((q1, 0.6), q3 - q1, 0.8, fc="#dbe9fb", ec=B, lw=3))
ax.plot([med, med], [0.6, 1.4], color=D, lw=3)
ax.plot([lo_w, q1], [1, 1], color=B, lw=3); ax.plot([q3, hi_w], [1, 1], color=B, lw=3)
for w in (lo_w, hi_w): ax.plot([w, w], [0.8, 1.2], color=B, lw=3)
ax.plot(30, 1, "o", ms=12, mfc="white", mec="#d1242f", mew=3)
ax.plot(x, np.full(len(x), -0.3), "o", color="#444", ms=9)          # the data, below
for f in (lo_f, hi_f): ax.axvline(f, color=G, ls="--", lw=1.5)
ax.text(lo_f + 0.3, 2.0, f"fence: Q1 − 1.5·IQR = {lo_f:g}", ha="left", color=G, fontsize=13)
ax.text(hi_f + 0.3, 2.0, f"fence: Q3 + 1.5·IQR = {hi_f:g}", ha="left", color=G, fontsize=13)
lab = dict(ha="center", fontsize=15, color=D)
ax.text(med, 1.5, "median = 5.5", **lab); ax.text(q1, 0.38, "Q1 = 3", **lab); ax.text(q3, 0.38, "Q3 = 8", **lab)
ax.annotate("", xy=(q1, 0.2), xytext=(q3, 0.2), arrowprops=dict(arrowstyle="<->", color=D, lw=2))
ax.text((q1 + q3) / 2, 0.02, "IQR = 5 (the middle 50%)", **lab)
ax.text(lo_w - 0.4, 1.0, "whisker ends at 1:\nthe lowest point\ninside the fence", ha="right", va="center", fontsize=13, color=B)
ax.text(hi_w + 0.4, 1.0, "whisker ends at 9:\nthe highest point\ninside the fence\n(not at 15.5!)", ha="left", va="center", fontsize=13, color=B)
ax.text(30, 1.25, "outlier", ha="center", fontsize=15, color="#d1242f")
ax.text(-7.5, -0.3, "the data:", va="center", fontsize=13, color="#444")
ax.set_xlim(-8, 32); ax.set_ylim(-0.5, 2.15); ax.set_yticks([])
for s in ("left", "right", "top"): ax.spines[s].set_visible(False)
ax.set_xticks(range(0, 31, 5))
plt.tight_layout(); plt.show()

## Box plots in Python

In [ ]:
np.random.seed(0)
scores = np.random.normal(loc=75, scale=10, size=100)
# patch_artist=True draws filled boxes, so boxprops can color them
# orientation='horizontal' needs Matplotlib 3.10+; older versions: vert=False
plt.boxplot(scores, orientation='horizontal', patch_artist=True,
            boxprops=dict(facecolor='red'))
plt.title("Distribution of Exam Scores (Matplotlib)")
plt.xlabel("Score")
plt.show()

## Bar chart in Python

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
data = pd.Series(['Dog', 'Cat', 'Dog', 'Bird', 'Cat', 'Dog'])

# Plot bar chart
data.value_counts().plot(kind='bar', color='green', edgecolor='black')
plt.title("Favorite Pets in a Survey")
plt.xlabel("Animal")
plt.ylabel("Frequency")
plt.show()

## Common mistake: categories in the wrong order

In [ ]:
# (on the slides, this code is hidden: only its output shows)
hours = pd.Series(["3-5", "Under 3", "6-8", "3-5", "12 or more", "9-11", "3-5", "6-8", "Under 3", "3-5"])
order = ["Under 3", "3-5", "6-8", "9-11", "12 or more"]
fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (title, s) in zip(axs, [("value_counts()", hours.value_counts()),
                               ("value_counts().sort_index()", hours.value_counts().sort_index()),
                               ("value_counts().reindex(order)", hours.value_counts().reindex(order))]):
    ax.bar(s.index, s.values, color="steelblue", edgecolor="black")
    ax.set_title(title, fontsize=14); ax.tick_params(axis="x", rotation=30)
axs[0].set_ylabel("Students")
plt.tight_layout(); plt.show()

## Pie chart: parts of one whole

In [ ]:
import matplotlib.pyplot as plt

# Example data
sizes = [40, 25, 20, 15]
labels = ['Apples', 'Bananas', 'Oranges', 'Berries']

# autopct labels each slice:
# '%1.1f%%' = 1 decimal, then a % sign
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
plt.title("Fruit Sales Distribution")
plt.show()

## Similar slices: the bar beats the pie

In [ ]:
companies = ['Co A','Co B','Co C','Co D','Co E','Co F','Co G','Other']
shares = [15, 14, 13, 12, 11, 10, 9, 16]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")

ax1.pie(shares, labels=companies, autopct='%1.1f%%', startangle=90)
ax1.set_title('Market Share (Pie): Hard to Compare')

ax2.barh(companies, shares, color='steelblue', edgecolor='black')
ax2.set_xlabel('Market Share (%)')
ax2.set_title('Market Share (Bar): Easy to Compare')
ax2.grid(axis='x', alpha=0.3)

plt.show()

## A good pie: one dominant slice

In [ ]:
categories = ['Programs','Operations','Fundraising','Admin']
percentages = [70, 15, 10, 5]
explode = (0.1, 0, 0, 0)   # emphasize the main slice

plt.figure(figsize=(7, 5))
plt.pie(percentages, labels=categories, autopct='%1.0f%%',
        startangle=90, explode=explode, textprops={'fontsize': 12})
plt.title('Where Your Donation Goes')
plt.show()

## 🥧 Your survey: pie or bar?

In [ ]:
survey = pd.read_csv("data/midquarter_survey_2026.csv")
survey["workload"].value_counts()

In [ ]:
more = survey["want_more_of"].str.split(";").explode()   # one row per pick
print(len(more), "picks from", len(survey), "students")

## Your survey, drawn: pie vs. bar

In [ ]:
# (on the slides, this code is hidden: only its output shows)
wl = survey["workload"].value_counts()
pct = (more.value_counts() / len(survey) * 100).sort_values()
fig, (a, b) = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1, 1.5]})
a.pie(wl, colors=[BLUE, ORANGE, GRAY], startangle=90, counterclock=False,
      autopct=lambda v: f"{v:.0f}%" if v > 5 else "", pctdistance=0.7,
      wedgeprops={"edgecolor": "white", "linewidth": 2}, textprops={"color": "white"})
a.legend([f"{k} ({v / len(survey):.0%})" for k, v in wl.items()], loc="lower center",
         bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=12)
a.set_title("Workload: a good pie")
b.barh(pct.index, pct.values, color=BLUE, height=0.6)
for yi, v in enumerate(pct.values):
    b.text(v + 0.8, yi, f"{v:.0f}%", va="center", color="#333")
b.set_xlim(0, 55); b.set_xlabel("% of students"); b.set_title("'More of ...': must be a bar chart")
b.spines["left"].set_visible(False); b.tick_params(axis="y", length=0)
plt.tight_layout(); plt.show()

## Method 1 in Python: `plt.hist`

In [ ]:
np.random.seed(0)
wait_times = np.concatenate([
    np.random.normal(35, 12, 7000),    # most patients
    np.random.normal(150, 30, 3000),   # a second group, waiting much longer
])
# a normal model allows a few negative waits: clip them to 0
wait_times = np.clip(wait_times, 0, None)

plt.figure(figsize=(10,5))
plt.hist(wait_times, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
plt.axvline(60, color='green', linestyle='--', linewidth=2, label='Target: 60 min')
plt.axvline(np.median(wait_times), color='black', linestyle=':', linewidth=2,
            label=f'Median: {np.median(wait_times):.0f} min')
plt.axvline(wait_times.mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean: {wait_times.mean():.0f} min')
plt.xlabel('Wait Time (minutes)')
plt.ylabel('Number of Patients')
plt.title('ER Wait Time Distribution')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Method 2 in Python: `plt.boxplot`

In [ ]:
np.random.seed(0)

departments = ['Triage', 'Lab', 'Radiology', 'Doctor\nConsult', 'Treatment', 'Discharge']
dept_sizes = [1500, 1500, 1500, 2500, 1500, 1500]

wait_times_by_dept = [
    np.random.normal(20, 5, dept_sizes[0]),     # Triage – short, consistent
    np.random.normal(60, 25, dept_sizes[1]),    # Lab – moderate variation
    np.random.normal(45, 15, dept_sizes[2]),    # Radiology – steady
    np.random.normal(120, 50, dept_sizes[3]),   # Doctor Consult – long, erratic
    np.random.normal(35, 10, dept_sizes[4]),    # Treatment – moderate
    np.random.normal(15, 4, dept_sizes[5])      # Discharge – quick
]

plt.figure(figsize=(12,6))
# tick_labels= needs Matplotlib 3.9+; older versions: labels=
bp = plt.boxplot(wait_times_by_dept, tick_labels=departments, patch_artist=True)
colors = ['lightgreen','yellow','lightgreen','red','lightgreen','lightgreen']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

plt.axhline(60, color='green', linestyle='--', label='Target: 60 min')
plt.ylabel("Wait Time (minutes)")
plt.title("ER Wait Times by Department")
plt.legend()
plt.show()

## 🎲 A game of chance

In [ ]:
values = np.array([-2, -1, 0, 1, 2])
probs  = np.array([0.3, 0.2, 0.0, 0.0, 0.5])
mean = (values * probs).sum()                    # expected winnings per play
var  = (values**2 * probs).sum() - mean**2
print(f"mean = {mean:.2f} dollars per play, variance = {var:.2f}")

## Two plays: every combination

In [ ]:
# (on the slides, this code is hidden: only its output shows)
P2 = np.outer(probs, probs)
fig, ax = plt.subplots(figsize=(6.4, 5.6))
ax.imshow(P2, cmap="Blues", origin="lower", vmin=0, vmax=0.3)
for i, a in enumerate(values):
    for j, b in enumerate(values):
        ax.text(j, i, f"{a + b:+d}\n{P2[i, j]:.2f}", ha="center", va="center", fontsize=12,
                color="white" if P2[i, j] > 0.18 else "#222")
ax.set_xticks(range(5), values); ax.set_yticks(range(5), values)
ax.set_xlabel("Play 2 outcome"); ax.set_ylabel("Play 1 outcome")
ax.set_title("top: total    bottom: probability", fontsize=14)
for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

## Two plays: every combination (answer)

In [ ]:
# (on the slides, this code is hidden: only its output shows)
P2 = np.outer(probs, probs)
fig, ax = plt.subplots(figsize=(6.4, 5.6))
ax.imshow(P2, cmap="Blues", origin="lower", vmin=0, vmax=0.3)
for i, a in enumerate(values):
    for j, b in enumerate(values):
        ax.text(j, i, f"{a + b:+d}\n{P2[i, j]:.2f}", ha="center", va="center", fontsize=12,
                color="white" if P2[i, j] > 0.18 else "#222")
ax.set_xticks(range(5), values); ax.set_yticks(range(5), values)
ax.set_xlabel("Play 2 outcome"); ax.set_ylabel("Play 1 outcome")
ax.set_title("top: total    bottom: probability", fontsize=14)
for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

## Another view: collapse the diagonals

In [ ]:
# (on the slides, this code is hidden: only its output shows)
totals = np.add.outer(values, values)
p2 = np.convolve(probs, probs)                     # P(total = -4 ... +4)
cmap = plt.get_cmap("coolwarm_r")
col = {t: cmap((t + 4) / 8) for t in range(-4, 5)}
fig, (a, b) = plt.subplots(1, 2, figsize=(15, 5.4), gridspec_kw={"width_ratios": [1, 1.5]})
for i in range(5):
    for j in range(5):
        t = totals[i, j]
        a.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, color=col[t], alpha=0.25 + 2.5 * P2[i, j],
                                  ec="white", lw=2))
        a.text(j, i, f"{t:+d}\n{P2[i, j]:.2f}", ha="center", va="center", fontsize=11)
a.set_xlim(-.5, 4.5); a.set_ylim(-.5, 4.5); a.set_aspect("equal")
a.set_xticks(range(5), values); a.set_yticks(range(5), values)
a.set_xlabel("Play 2"); a.set_ylabel("Play 1"); a.set_title("Same total = same diagonal")
for s in a.spines.values(): s.set_visible(False)
ts = np.arange(-4, 5)
b.bar(ts, p2, color=[col[t] for t in ts], edgecolor="#555", width=0.8)
for t, p in zip(ts, p2):
    if p > 0: b.text(t, p + 0.006, f"{p:.2f}", ha="center", fontsize=12)
b.set_xticks(ts); b.set_xlabel("Total winnings after 2 plays"); b.set_ylabel("Probability")
b.set_title("Each diagonal becomes one bar"); b.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## Adding up the diagonals is convolution

In [ ]:
p2 = np.convolve(probs, probs)                 # totals -4, -3, ..., +4
for t, p in zip(range(-4, 5), p2):
    print(f"P(total = {t:+d}) = {p:.2f}")

## The EE view: flip, slide, multiply, add

In [ ]:
# (on the slides, this code is hidden: only its output shows)
def pmf(k):                                       # p(k) for any integer k (0 outside -2..2)
    return probs[k + 2] if -2 <= k <= 2 else 0.0
K = np.arange(-6, 7)
def slide_panel(ax, t, small=False):
    a = np.array([pmf(k) for k in K]); b = np.array([pmf(t - k) for k in K])
    nz_a, nz_b = a > 0, b > 0
    ax.vlines(K[nz_a] - 0.12, 0, a[nz_a], color=BLUE, lw=3)
    ax.plot(K[nz_a] - 0.12, a[nz_a], "o", color=BLUE, ms=7, label="p(k)")
    ax.vlines(K[nz_b] + 0.12, 0, b[nz_b], color=ORANGE, lw=3)
    ax.plot(K[nz_b] + 0.12, b[nz_b], "s", color=ORANGE, ms=7, label="p(t − k), flipped & slid")
    both = nz_a & nz_b
    for k in K[both]:
        ax.axvspan(k - 0.4, k + 0.4, color="#f2d24b", alpha=0.35, lw=0)
    terms = " + ".join(f"{pmf(k):.1f}·{pmf(t - k):.1f}" for k in K[both]) or "no overlap"
    ax.set_title(f"t = {t:+d}:   {terms}  =  {sum(pmf(k) * pmf(t - k) for k in K):.2f}",
                 fontsize=14 if small else 15)
    ax.axhline(0, color="#888", lw=1); ax.set_ylim(0, 0.62); ax.set_xticks(K)
    ax.set_yticks([0, 0.25, 0.5]); ax.grid(axis="y", alpha=0.25)
fig, axs = plt.subplots(2, 3, figsize=(15, 5.0), sharey=True)
for ax, t in zip(axs.flat, [-4, -3, 0, 1, 2, 4]):
    slide_panel(ax, t, small=True)
axs[0, 0].legend(loc="upper left", fontsize=10, frameon=False)
for ax in axs[1]: ax.set_xlabel("k")
plt.tight_layout(); plt.show()

## Watch it slide

In [ ]:
# (on the slides, this code is hidden: only its output shows)
# Animation: the flipped PMF slides past p(k); each overlap sum becomes one stem of P(S2 = t).
from matplotlib.animation import FuncAnimation, PillowWriter
import os
os.makedirs("figs", exist_ok=True)
T = np.arange(-6, 7)
res = np.array([sum(pmf(k) * pmf(t - k) for k in K) for t in T])
fig, (ax, bx) = plt.subplots(2, 1, figsize=(12, 6.2), gridspec_kw={"height_ratios": [1, 1]})
def draw(i):
    t = T[i]; ax.clear(); bx.clear()
    slide_panel(ax, t); ax.legend(loc="upper left", fontsize=11, frameon=False)
    done = T <= t
    bx.vlines(T[done], 0, res[done], color="#555", lw=3); bx.plot(T[done], res[done], "o", color="#555", ms=7)
    bx.plot(t, res[i], "o", color="#e34948", ms=12, zorder=3)
    bx.set_xlim(K[0] - 0.5, K[-1] + 0.5); bx.set_ylim(0, 0.36); bx.set_xticks(K)
    bx.axhline(0, color="#888", lw=1); bx.grid(axis="y", alpha=0.25)
    bx.set_xlabel("t (total after 2 plays)"); bx.set_ylabel("P(S₂ = t)")
    fig.tight_layout()
frames = [0, 0] + list(range(len(T))) + [len(T) - 1] * 3
FuncAnimation(fig, draw, frames=frames).save("figs/convolution_slide.gif", writer=PillowWriter(fps=1))
plt.close(fig)
show_gif("figs/convolution_slide.gif", width="78%")

## Keep playing: the total after n plays

In [ ]:
# (on the slides, this code is hidden: only its output shows)
# Build the animation once (cached by freeze). Repeated convolution gives the exact PMF of n plays.
from matplotlib.animation import FuncAnimation, PillowWriter
import os
os.makedirs("figs", exist_ok=True)
NMAX = 30
pmfs = [probs]
for _ in range(NMAX - 1):
    pmfs.append(np.convolve(pmfs[-1], probs))
ns = np.arange(1, NMAX + 1)
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [2, 1]})
def draw(k):
    n, p = ns[k], pmfs[k]
    s = np.arange(-2 * n, 2 * n + 1)
    ax.clear(); ax2.clear()
    ax.bar(s, p, width=0.85, color=BLUE, label=f"exact PMF of the total of {n} plays")
    x = np.linspace(-2 * n - 1, 2 * n + 1, 400)
    ax.plot(x, np.exp(-(x - n * mean) ** 2 / (2 * n * var)) / np.sqrt(2 * np.pi * n * var),
            color="#e34948", lw=2.5, label="Gaussian, same mean and variance")
    half = min(2 * n + 1.5, 4.2 * np.sqrt(n * var))
    ax.set_xlim(n * mean - half, n * mean + half); ax.set_ylim(0, p.max() * 1.3)
    ax.set_xlabel("Total winnings after n plays ($)"); ax.set_ylabel("Probability")
    ax.set_title(f"n = {n}"); ax.legend(loc="upper left", fontsize=11, frameon=False)
    ax2.plot(ns, ns * mean, "o-", ms=3, color=AQUA, label=f"mean = {mean:.1f} n")
    ax2.plot(ns, ns * var, "s-", ms=3, color=ORANGE, label=f"variance = {var:.2f} n")
    ax2.axvline(n, color="gray", ls="--")
    ax2.set_xlabel("n (number of plays)"); ax2.set_title("Mean and variance grow linearly")
    ax2.legend(fontsize=11, frameon=False)
    fig.tight_layout()
frames = [0, 0, 1, 1, 2, 2, 3, 3, 4, 5, 6, 8, 10, 13, 17, 22, 29, 29, 29]
FuncAnimation(fig, draw, frames=frames).save("figs/clt_convolution.gif", writer=PillowWriter(fps=1.2))
draw(len(ns) - 1); fig.savefig("figs/clt_convolution_n30.png", dpi=110)   # static frame for PDF handouts
plt.close(fig)
show_gif("figs/clt_convolution.gif", width="95%")